# Store Sales Machine Learning
## Machine Learning & Prediction

* **Goal:** Develop a machine learning model to predict sales for individual stores and product families based on historical sales data, promotions, calendar features, and other relevant variables.

In [3]:
# Import libraies
from pathlib import Path
import pandas as pd

data_path = Path('../data/train_ml.csv')
train = pd.read_csv(data_path, parse_dates=['date'])

train.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,year,month,day,day_of_week,is_weekend,quarter,is_special_date,day_of_year
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,1746.0,93.14,2013,1,1,1,0,1,1,1
1,1194,2013-01-01,42,CELEBRATION,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
2,1193,2013-01-01,42,BREAD/BAKERY,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
3,1192,2013-01-01,42,BOOKS,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1
4,1191,2013-01-01,42,BEVERAGES,0.0,0,Cuenca,Azuay,D,2,1053.0,93.14,2013,1,1,1,0,1,1,1


In [5]:
# Inspection
print(f'Rows: {train.shape[0]} -- Cols: {train.shape[1]}')

print(train.columns.tolist())

print(train[['date', 'store_nbr', 'family', 'sales']].head())

Rows: 3000888 -- Cols: 20
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'transactions', 'dcoilwtico', 'year', 'month', 'day', 'day_of_week', 'is_weekend', 'quarter', 'is_special_date', 'day_of_year']
        date  store_nbr        family  sales
0 2013-01-01          1    AUTOMOTIVE    0.0
1 2013-01-01         42   CELEBRATION    0.0
2 2013-01-01         42  BREAD/BAKERY    0.0
3 2013-01-01         42         BOOKS    0.0
4 2013-01-01         42     BEVERAGES    0.0


In [6]:
train['date'].min(), train['date'].max()

(Timestamp('2013-01-01 00:00:00'), Timestamp('2017-08-15 00:00:00'))

In [7]:
# Sort data chronologically
train = train.sort_values('date').reset_index(drop=True)

validation_start = '2017-08-01'

# Split data by date
train_data = train[train['date'] < validation_start].copy()
valid_data = train[train['date'] >= validation_start].copy()

# Check split
print('Trainings period:')
print(f' {train_data['date'].min()} to {train_data['date'].max()}')
print('Validation period:')
print(f' {valid_data['date'].min()} to {valid_data['date'].max()}')

Trainings period:
 2013-01-01 00:00:00 to 2017-07-31 00:00:00
Validation period:
 2017-08-01 00:00:00 to 2017-08-15 00:00:00


In [9]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate average sales for each store and product family
sales_mean = (
    train_data
    .groupby(['store_nbr', 'family'])['sales']
    .mean()
)

# Create predition for vald.perido
valid_prediction = valid_data.set_index(
    ['store_nbr', 'family']
).index.map(sales_mean)

# Replace missing pred. with overall training average
valid_prediction = np.where(
    pd.isna(valid_prediction), 
    train_data['sales'].mean(), 
    valid_prediction
)

# Evaluate baseline
mae = mean_absolute_error(valid_data['sales'], valid_prediction)
rmse = np.sqrt(mean_squared_error(valid_data['sales'], valid_prediction))

print(f'Baseline MAE {mae:.2f}')
print(f'Baseline RMSE: {rmse:.2f}')


Baseline MAE 143.64
Baseline RMSE: 506.74
